# Session 4: Bootcamp: Forecasting Bike-Sharing Demand

## Overview

This bootcamp puts Sessions 1-3 together in one end-to-end project. Capital Bikeshare, in Washington D.C., moves
bikes between stations by truck every night so that there are bikes where people will want them. To plan those
moves it needs a forecast of **how many bikes will be rented in each hour**, given the calendar and the weather
forecast. You will build that forecast the way a real project goes:

1. **Understand the problem and the data**, including which columns must *not* be used.
2. **Split the data the way the model will be used:** train on the past, test on the future.
3. **Explore** to find the patterns a model has to capture.
4. **Start with a baseline**, then a linear model, then gradient boosting.
5. **Choose with cross-validation** that respects time; test once.
6. **Analyse the errors** and explain the model, so you know when not to trust it.

By the end, you will be able to:

- Spot target leakage in a dataset's columns.
- Split time-ordered data by time, and cross-validate with `TimeSeriesSplit`.
- Show why a shuffled split overestimates performance on time-ordered data.
- Build a `ColumnTransformer` pipeline with one-hot encoding and a log-transformed target.
- Analyse errors by hour and by day, and explain what a forecast cannot know.

**Prerequisites:** Sessions 1-3.

**Dataset:** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). 17,379 hours of
Capital Bikeshare rentals in 2011 and 2012, with the date, hour, season, holiday and working-day flags, the weather
situation, temperature, "feels-like" temperature, humidity and wind speed, and the number of rentals by casual and
registered users. It is downloaded automatically by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models **← you are here** |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** How accurately can we forecast hourly bike rentals for a future quarter, and where
does the forecast go wrong?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| scikit-learn | Pipelines, encoding, models, time-series cross-validation |
| pandas | Tables and dates |
| numpy | Arrays, log transform |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from importlib.metadata import version

for package in ["ucimlrepo", "scikit-learn", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data and read the column descriptions

UCI datasets come with a description of every column. Read it before doing anything else; it is the cheapest
way to avoid a wrong model.

In [ ]:
from ucimlrepo import fetch_ucirepo

bikes = fetch_ucirepo(id=275)
print(bikes.variables[["name", "role", "type", "description"]].to_string(max_colwidth=70))

hours = bikes.data.original
hours["dteday"] = pd.to_datetime(hours["dteday"])
hours = hours.sort_values(["dteday", "hr"]).reset_index(drop=True)
print(f"\n{len(hours):,} hours from {hours['dteday'].min():%Y-%m-%d} to {hours['dteday'].max():%Y-%m-%d}")

**Read the output:** the target is `cnt`, the number of rentals in the hour. Note how the weather is stored:
`temp`, `atemp`, `hum` and `windspeed` are **scaled to 0-1** (for example, temp = (°C + 8) / 47), and `weathersit`
is a category from 1 (clear) to 4 (heavy rain or snow). Two columns have the role **Other**: `casual` and
`registered`. Step 3 explains why that matters.

## Step 3: Find the leak

**Target leakage** means an input that contains the answer, or is only known after the answer. A model trained
with it looks brilliant in testing and is useless in practice.

In [ ]:
print("casual + registered == cnt in every row:", (hours["casual"] + hours["registered"] == hours["cnt"]).all())

**Read the output:** `True`. The total is literally the sum of these two columns, which are counted *during*
the hour we want to forecast. A model given them would just add them up and score perfectly, and could never be
used: when the nightly plan is made, nobody knows tomorrow's casual and registered rentals.

We use only inputs known in advance: the calendar, and the weather (in practice, the weather *forecast*). We also
drop `instant` (a row number) and keep `dteday` only for splitting and analysis.

In [ ]:
CATEGORICAL = ["season", "mnth", "hr", "weekday", "weathersit"]
FLAGS = ["yr", "holiday", "workingday"]
WEATHER = ["temp", "atemp", "hum", "windspeed"]
FEATURES = CATEGORICAL + FLAGS + WEATHER

## Step 4: Split by time

The forecast will always be made for the **future**, from data about the past. The test must copy that: we train
on January 2011 to September 2012 and hold out **the last quarter, October to December 2012**. A random split would
put, say, 8 a.m. and 10 a.m. of the same day in training and 9 a.m. in the test set, an easy interpolation the real
forecast never gets to make. Step 9 measures how much this matters.

In [ ]:
is_test = hours["dteday"] >= "2012-10-01"
train, test = hours[~is_test], hours[is_test]
X_train, y_train = train[FEATURES], train["cnt"]
X_test, y_test = test[FEATURES], test["cnt"]
print(f"train: {len(train):,} hours ({train['dteday'].min():%b %Y} - {train['dteday'].max():%b %Y})")
print(f"test:  {len(test):,} hours ({test['dteday'].min():%b %Y} - {test['dteday'].max():%b %Y})")

## Step 5: Explore the patterns

Three questions a forecaster would ask first: how does demand vary over the day, over the year, and with the
weather? We explore the **training** period only.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
train.groupby(["hr", "workingday"])["cnt"].mean().unstack().rename(
    columns={0: "weekend / holiday", 1: "working day"}).plot(ax=axes[0])
axes[0].set_title("Average rentals by hour")
axes[0].set_ylabel("rentals per hour")

monthly = train.groupby([train["dteday"].dt.year, "mnth"])["cnt"].mean().unstack(0)
monthly.plot(ax=axes[1], marker="o")
axes[1].set_title("Average rentals by month")

train.groupby("weathersit")["cnt"].mean().plot.bar(ax=axes[2])
axes[2].set_title("Average rentals by weather (1 = clear, 4 = heavy rain/snow)")
plt.tight_layout()
plt.show()

print(train.groupby("weathersit")["cnt"].agg(["count", "mean"]).round(0))
print(f"\naverage rentals per hour, Jan-Sep: 2011 = {train.query('yr == 0 and mnth <= 9')['cnt'].mean():.0f}, "
      f"2012 = {train.query('yr == 1')['cnt'].mean():.0f}")

**Read the output:**

- **Hour of day:** on working days there are two sharp **commuting peaks**, around 8 a.m. and 5-6 p.m.; on
  weekends and holidays there is one broad afternoon hump. The model needs the hour *and* the type of day together:
  an interaction.
- **Season and growth:** demand is low in winter and high from May to October, and 2012 is far busier than 2011
  for the same months (240 against 144 rentals per hour, January to September): the scheme is growing. The
  test quarter is the busiest October-December ever, which the model must be able to represent.
- **Weather:** average rentals fall from 201 per hour when clear (1) to 168 when misty (2) and 108 in light rain or
  snow (3). Category 4 (heavy rain or snow) occurs in only 3 training hours, far too few to learn from.

## Step 6: A baseline

The baseline is the forecast you would make with a spreadsheet: **the average for that hour on that type of
day** (working or not), from the training data. Every model has to beat it to be worth its complexity.

In [ ]:
def report(name, y_true, y_pred):
    print(f"{name:34s} MAE = {mean_absolute_error(y_true, y_pred):5.1f}   "
          f"RMSE = {mean_squared_error(y_true, y_pred) ** 0.5:5.1f} rentals per hour")


profile = train.groupby(["hr", "workingday"])["cnt"].mean()
baseline = profile.reindex(pd.MultiIndex.from_frame(test[["hr", "workingday"]])).to_numpy()
report("baseline: hour x working-day average", y_test, baseline)
print(f"average rentals per hour in the test quarter: {y_test.mean():.0f}")

**Read the output:** the baseline misses by 80 rentals per hour on average (RMSE 119.6), against an average of
219 per hour in the test quarter. It knows the daily rhythm but nothing about weather, season or
growth.

## Step 7: A linear model, done properly

Two things make this data awkward for linear regression:

1. `hr`, `mnth`, `weekday`, `season` and `weathersit` are **categories coded as numbers**. Hour 17 is not "17 times
   hour 1". `OneHotEncoder` turns each category into its own 0/1 column, so the model can learn a separate effect
   for every hour.
2. Counts are **skewed** and never negative, and their spread grows with their level. Modelling **log(1 + count)**
   and converting back makes the errors more even and the effects multiplicative ("rain cuts demand by 40%")
   rather than additive. `TransformedTargetRegressor` does the transform and the conversion back for us.

A `ColumnTransformer` applies the encoder to the categorical columns and passes the others through unchanged.

In [ ]:
encode = ColumnTransformer([("onehot", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)],
                           remainder="passthrough")
linear = TransformedTargetRegressor(make_pipeline(encode, LinearRegression()),
                                    func=np.log1p, inverse_func=np.expm1)

## Step 8: Gradient boosting

From Session 3: trees find interactions (hour × working day) and curves (temperature) by themselves, and do not
need one-hot encoding, because a tree can split hours into "7-9 a.m." versus the rest directly.
`HistGradientBoostingRegressor` is scikit-learn's fast implementation for larger datasets: it bins each input
into at most 255 values before searching for splits.

In [ ]:
boost = HistGradientBoostingRegressor(random_state=0)

## Step 9: Cross-validate the way the model will be used

`TimeSeriesSplit` cross-validates **forwards in time**: fold 1 trains on the earliest stretch and validates on the
next one; each later fold trains on everything before its validation stretch. For comparison, we also run an
ordinary shuffled `KFold`, the default you would reach for without thinking.

In [ ]:
time_cv = TimeSeriesSplit(n_splits=5)
shuffled_cv = KFold(n_splits=5, shuffle=True, random_state=0)


def cv_rmse(model, cv):
    return -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")


for name, model in [("linear (one-hot, log target)", linear), ("gradient boosting", boost)]:
    forward = cv_rmse(model, time_cv)
    print(f"{name:30s} time-series CV RMSE = {forward.mean():5.1f}   folds: {forward.round(0)}")
print(f"{'gradient boosting':30s} shuffled CV RMSE    = {cv_rmse(boost, shuffled_cv).mean():5.1f}")

**Read the output:**

- With forward-in-time validation, gradient boosting (RMSE 76.4) clearly beats the linear model (119.2). We
  choose gradient boosting.
- The fold errors vary a lot (from 48 to 100 for gradient boosting), because each fold covers a different stretch
  of the year: quiet winter months are easier to forecast in absolute terms than busy summer ones.
- **The shuffled CV says 39.5: half the real error.** With shuffled folds, every validation hour has its
  neighbours from the same day in training, so the model effectively already knows that day's weather and
  demand. Had we trusted it, we would have promised the operations team a forecast twice as good as the one they
  would get.

## Step 10: Test on the last quarter of 2012

Fit on all the training data and forecast October-December 2012. A count cannot be negative, so predictions are
clipped at zero.

In [ ]:
linear.fit(X_train, y_train)
boost.fit(X_train, y_train)
test = test.assign(forecast=np.clip(boost.predict(X_test), 0, None))

report("baseline: hour x working-day average", y_test, baseline)
report("linear (one-hot, log target)", y_test, linear.predict(X_test))
report("gradient boosting", y_test, test["forecast"])

**Read the output:** gradient boosting misses by 44 rentals per hour on average (RMSE 67.4), a little better than
its time-series CV estimate (76.4), and with almost half the baseline's error. The linear model, despite the
one-hot encoding and log target, is barely better than the baseline (MAE 72.7, RMSE 110.0): it gives every hour
the same effect on working days and weekends, so it cannot make commuter peaks on weekdays only without more
hand-made features.

## Step 11: Where does the forecast go wrong?

An average error hides where the errors are. Look at them by hour of day, then by date.

In [ ]:
test = test.assign(error=test["cnt"] - test["forecast"])
by_hour = test.groupby("hr")["error"].apply(lambda e: np.sqrt(np.mean(e ** 2)))
print("RMSE by hour of day:")
print(by_hour.round(0).to_frame("RMSE").T.to_string())

daily = test.groupby("dteday")[["cnt", "forecast"]].sum()
daily["error"] = daily["cnt"] - daily["forecast"]
print("\nThe six worst days (daily totals):")
print(daily.reindex(daily["error"].abs().sort_values(ascending=False).index).head(6).round(0))

week = test[(test["dteday"] >= "2012-11-19") & (test["dteday"] < "2012-11-26")]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(week)), week["cnt"], label="actual")
ax.plot(range(len(week)), week["forecast"], label="forecast")
ax.set_xticks(range(0, len(week), 24), week["dteday"].dt.strftime("%a %d %b").iloc[::24])
ax.set_ylabel("rentals per hour")
ax.set_title("Thanksgiving week 2012")
ax.legend()
plt.show()

**Read the output:**

- **By hour:** errors are largest at the commuting peaks (RMSE 135 at 8 a.m., 124 and 116 at 5 and 6 p.m.) and
  smallest overnight (11-24 between 1 and 5 a.m.). Peaks are where demand is highest, and where a missed forecast
  leaves stations empty.
- **By day:** the six worst days are all in the **holiday season**: Thanksgiving (Thursday 22 November) and the two
  days after it, and 24-26 December. On each, the model forecast far too many rentals: about 2,300-2,700 too many on
  each Thanksgiving day, and five times the actual number on 26 December. The plot shows good forecasts from
  Monday to Wednesday afternoon, then forecasts well above the actual rentals from Wednesday evening on, as people
  leave for the holiday. On Friday the model expects commuter peaks that never come.

Why? Thanksgiving and Christmas Day are flagged as holidays, so the model treats them like an ordinary weekend
day, when many people ride for leisure; on these days most people stay at home. Christmas Eve, 26 December and the
day after Thanksgiving are not flagged at all, so the model expects a normal working day with commuter peaks. Each
of these days appears only **once** in the training data (in 2011), too rarely to learn from. This is a limit of
the data, not of the algorithm, and the fix is outside the model: an operator overriding the forecast for known
holidays, or an extra "holiday season" input.

## Step 12: What does the model rely on?

Permutation importance on the test quarter (Session 3, Step 8).

In [ ]:
perm = permutation_importance(boost, X_test, y_test, scoring="neg_root_mean_squared_error",
                              n_repeats=10, random_state=0)
print(pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False).round(1)
      .to_frame("RMSE increase").T.to_string())

**Read the output:** **hour of day** dominates (scrambling it adds about 170 rentals per hour to the error), then
**working day**, then **temperature**. Weather situation and humidity matter less.

`yr` shows an importance of **0**. That does not mean the growth between 2011 and 2012 is unimportant. Every test
hour is in 2012, so scrambling `yr` *within the test set* changes nothing. Permutation importance can only measure
what varies in the data you compute it on.

## Bootcamp summary

| Step | Decision | Why |
| --- | --- | --- |
| Columns | Drop `casual`, `registered`, `instant` | `casual + registered = cnt`: target leakage |
| Split | Last quarter as test | The forecast is always for the future |
| Validation | `TimeSeriesSplit` | Shuffled folds gave half the real error |
| Model | Gradient boosting | Time-series CV RMSE about 76 against 119 for linear and 120 for the baseline |
| Result | Test MAE 44, RMSE 67 rentals per hour | In line with the CV estimate (76) |
| Known limit | Holidays and the holiday season | Seen once in training; override or add inputs |

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Using `casual` and `registered` as inputs | Near-perfect test score, useless forecast | Read the column descriptions; keep only inputs known in advance |
| Random split or shuffled CV on time-ordered data | Error estimate about half the real one | Split by time; use `TimeSeriesSplit` |
| Treating hour or month codes as plain numbers in a linear model | One slope for "hour", which cannot make two peaks | `OneHotEncoder` inside a `ColumnTransformer` |
| Forgetting to undo a log transform | Forecasts on the log scale | `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` |
| Reporting only the overall RMSE | Peak hours and holidays hidden in the average | Break errors down by hour and by date |
| Reading zero permutation importance as "useless" | The input may not vary in the evaluation data | Check what varies before interpreting |

## Exercises

- [ ] Add an input `days_since_start` (days since 1 January 2011). Does it help the linear model capture growth?
  Does it help gradient boosting? (Think about how trees extrapolate beyond the training range.)
- [ ] Add a "holiday season" flag for 20 November to 31 December and re-run Steps 10 and 11. Do the worst days improve?
- [ ] Forecast `casual` and `registered` with two separate models and add the forecasts. Is the sum better than
  forecasting `cnt` directly? Why might it be?
- [ ] Use `loss="poisson"` in `HistGradientBoostingRegressor`, which is designed for counts. Compare its
  time-series CV RMSE.

This ends Part 1. Next, Session 5 starts classification: predicting a category instead of a number, beginning,
like Session 1, with a model built from scratch: logistic regression by gradient descent.